In [1]:
import os


import pandas as pd
import matplotlib as plt


import pandas as pd
import matplotlib.pyplot as plt

df_clean = pd.read_csv("HealthConnect_clean.csv")

In [2]:
df_clean

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,...,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome,lead_time_group,previous_appointment_group,distance_group,waiting_time_group
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2025-02-06,2025-02-18,Tuesday,Afternoon,...,0,Yes,WhatsApp,19.3,29.0,No-Show,0-14 days,1-2 appointments,15+ km,16-30 min
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2026-02-25,2026-02-27,Friday,Morning,...,0,Yes,SMS,14.3,42.0,Attended,0-14 days,5+ appointments,10-15 km,31-45 min
2,HC-00003,P-1366,Female,50,45-54,General Consultation,2025-11-16,2025-12-24,Wednesday,Morning,...,1,Yes,SMS,11.4,11.0,No-Show,30-44 days,5+ appointments,10-15 km,0-15 min
3,HC-00004,P-1031,Male,59,55-64,Follow-up,2025-07-18,2025-08-28,Thursday,Evening,...,1,Yes,SMS,7.4,35.0,Attended,30-44 days,3-4 appointments,5-10 km,31-45 min
4,HC-00005,P-1458,Female,34,25-34,Follow-up,2025-07-09,2025-08-25,Monday,Afternoon,...,1,Yes,Email,5.6,27.0,No-Show,45-60 days,3-4 appointments,5-10 km,16-30 min
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,HC-04996,P-0007,Male,49,45-54,Diagnostic Test,2025-03-29,2025-04-26,Saturday,Morning,...,1,Yes,Email,4.4,35.0,No-Show,15-29 days,5+ appointments,0-5 km,31-45 min
4996,HC-04997,P-1501,Female,52,45-54,Follow-up,2025-12-14,2026-02-12,Thursday,Evening,...,2,Yes,WhatsApp,12.6,24.0,Attended,45-60 days,5+ appointments,10-15 km,16-30 min
4997,HC-04998,P-0315,Male,59,55-64,General Consultation,2026-03-03,2026-03-24,Tuesday,Afternoon,...,0,No,NaN,22.8,28.0,Attended,15-29 days,5+ appointments,15+ km,16-30 min
4998,HC-04999,P-1262,Male,47,45-54,General Consultation,2025-12-03,2026-01-06,Tuesday,Morning,...,1,No,NaN,12.5,16.0,Attended,30-44 days,3-4 appointments,10-15 km,16-30 min


# Investigating Booking lead time

In [3]:
lead_time_type = (
    df_clean
    .groupby(["appointment_type", "lead_time_group"])["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .unstack()
    .round(2)
)

lead_time_type

lead_time_group,0-14 days,15-29 days,30-44 days,45-60 days
appointment_type,,,,
Diagnostic Test,30.88,43.70,51.83,68.99
Follow-up,32.97,43.64,57.34,70.57
General Consultation,29.25,42.75,49.81,64.84
Specialist Consultation,29.57,42.49,53.36,65.89


##### The strong relationship between booking lead time and no-show behaviour is not limited to a single appointment type. No-show rates consistently increase as booking lead time increases across all four appointment types.

In [4]:
lead_time_history = (
    df_clean
    .groupby(["previous_appointment_group", "lead_time_group"])["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .unstack()
    .round(2)
)

lead_time_history

lead_time_group,0-14 days,15-29 days,30-44 days,45-60 days
previous_appointment_group,,,,
0 appointments,20.37,37.70,43.28,68.33
1-2 appointments,30.45,39.28,51.42,63.94
3-4 appointments,32.45,44.90,53.70,67.65
5+ appointments,29.36,48.25,56.78,72.41


##### Long booking lead time appears to be an important no-show risk factor independently of previous appointment history.
##### The association between long booking lead time and no-show behaviour persists across all levels of previous appointment history.

In [5]:
lead_time_distance = (
    df_clean
    .groupby(["distance_group", "lead_time_group"])["appointment_outcome"]
    .apply(lambda y: (y == "No-Show").mean()*100).round(2)
    .unstack()
)

lead_time_distance

lead_time_group,0-14 days,15-29 days,30-44 days,45-60 days
distance_group,,,,
0-5 km,25.77,38.28,56.39,66.22
10-15 km,32.73,42.96,51.32,66.55
15+ km,36.78,50.22,56.82,73.00
5-10 km,28.61,42.24,49.33,65.38


##### The 45–60 day booking group has a no-show rate above 65% in every distance category.

In [6]:
lead_time_reminder = (
    df_clean
    .groupby(["reminder_sent", "lead_time_group"])["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean()*100).round(2)
    .unstack()
    
)
lead_time_reminder

lead_time_group,0-14 days,15-29 days,30-44 days,45-60 days
reminder_sent,,,,
No,32.32,46.46,53.74,73.46
Yes,29.98,41.69,52.46,65.05


##### The lower no-show rate associated with reminders persists across all booking lead-time groups and is largest among appointments booked 45–60 days in advance. We still cannot conclude that reminders cause the reduction. There may be other differences between appointments that received and did not receive reminders.

In [7]:
long_lead = df_clean[df_clean["lead_time_group"] == "45-60 days"]
# Total appointments booked 45–60 days in advance.

long_lead_appointments = len(long_lead)
long_lead_no_shows = (long_lead["appointment_outcome"] == "No-Show").sum()
# Number of those appointments that became no-shows.
share_of_all_no_shows = (
    long_lead_no_shows / (df_clean["appointment_outcome"] == "No-Show").sum()
) * 100
# Percentage of all HealthConnect no-shows that came from the 45–60 day group.

long_lead_appointments, long_lead_no_shows, round(share_of_all_no_shows, 2)

(1251, np.int64(841), np.float64(34.71))

##### Appointments booked 45–60 days in advance represent a disproportionately high no-show burden, accounting for 34.71% of all no-shows despite representing only 25.02% of appointments.

In [8]:
long_lead_rate = (
    long_lead_no_shows / long_lead_appointments
) * 100

difference_from_overall = long_lead_rate - 48.46

round(long_lead_rate, 2), round(difference_from_overall, 2)

(np.float64(67.23), np.float64(18.77))

##### Long booking lead time is not only the strongest Week 5 pattern; it also represents a major operational risk, with the 45–60 day group showing a 67.23% no-show rate—18.77 percentage points above the overall rate.

### Validating Booking lead Time

In [9]:
from scipy.stats import chi2_contingency

lead_time_test = pd.crosstab(
    df_clean["lead_time_group"],
    df_clean["appointment_outcome"] == "No-Show"
)

chi2, p_value, dof, expected = chi2_contingency(lead_time_test)

chi2, p_value

(np.float64(359.3752285032348), np.float64(1.3918045808651911e-77))

##### The relationship between booking lead time and no-show behaviour is statistically significant in the HealthConnect dataset (χ² = 359.38, p < 0.001). No-show rates increase substantially as booking lead time increases, with the 45–60 day group reaching 67.23%.

### Validating Previous Appointment History

In [10]:
history_test = pd.crosstab(
    df_clean["previous_appointment_group"],
    df_clean["appointment_outcome"] == "No-Show"
)

history_test

appointment_outcome,False,True
previous_appointment_group,,
0 appointments,138,104
1-2 appointments,1011,869
3-4 appointments,978,969
5+ appointments,450,481


##### Notice the pattern: as previous appointment history increases, the no-show count becomes increasingly close to, and eventually exceeds, the attended/non-no-show count. But counts alone don't tell us whether this relationship is statistically meaningful.

In [11]:
chi2, p_value, dof, expected = chi2_contingency(history_test)

chi2, p_value

(np.float64(11.844253232425599), np.float64(0.007936178958281382))

##### Previous appointment history was statistically associated with no-show behaviour (χ² = 11.84, p = 0.0079), supporting the Week 5 finding. However, the evidence is substantially weaker than for booking lead time (χ² = 359.38), suggesting that booking lead time remains the more prominent observed risk factor.

### Validating Distance to the clinic

In [12]:
distance_test = pd.crosstab(
    df_clean["distance_group"],
    df_clean["appointment_outcome"] == "No-Show"
)

distance_test

appointment_outcome,False,True
distance_group,,
0-5 km,619,537
10-15 km,583,549
15+ km,427,503
5-10 km,905,787


##### The important pattern is that the 15+ km group has noticeably more no-shows relative to its total appointments.

In [13]:
chi2, p_value, dof, expected = chi2_contingency(distance_test)

chi2, p_value

(np.float64(16.210345654919458), np.float64(0.001026756592585705))

##### Distance to the clinic was statistically associated with no-show behaviour (χ² = 16.21, p = 0.0010), supporting the Week 5 finding that patients living farther from the clinic had higher observed no-show rates. However, the strength of evidence was considerably lower than for booking lead time.

### Validating reminder status

In [14]:
reminder_test = pd.crosstab(
    df_clean["reminder_sent"],
    df_clean["appointment_outcome"] == "No-Show"
)

reminder_test

appointment_outcome,False,True
reminder_sent,,
No,664,702
Yes,1913,1721


In [15]:
chi2, p_value, dof, expected = chi2_contingency(reminder_test)

chi2, p_value

(np.float64(6.3037764278786295), np.float64(0.012048104033231888))

##### Appointments receiving reminders had a lower observed no-show rate, and reminder status was statistically associated with no-show behaviour (χ² = 6.30, p = 0.012). However, the observational nature of the data prevents us from concluding that reminders caused the reduction.

##### We already know the 45–60 day group is especially important. Now let's quantify how many additional no-shows occurred in this group compared with what we would expect if it had the overall clinic no-show rate.

In [16]:
expected_long_lead_no_shows = (
    long_lead_appointments * 0.4846
)

excess_no_shows = (
    long_lead_no_shows - expected_long_lead_no_shows
)

round(expected_long_lead_no_shows, 2), round(excess_no_shows, 2)

(606.23, np.float64(234.77))

##### Appointments booked 45–60 days in advance generated approximately 235 more no-shows than expected when compared with the clinic's overall no-show rate.


##### let's investigate whether we can identify an even more specific high-risk combination using our strongest factors. We'll start with lead time + distance, because we already found that: 45–60 days + 15+ km = 73.00% no-show rate.

In [17]:
high_risk_segment = df_clean[
    (df_clean["lead_time_group"] == "45-60 days") &
    (df_clean["distance_group"] == "15+ km")
]

len(high_risk_segment), (
    high_risk_segment["appointment_outcome"] == "No-Show"
).sum()

(237, np.int64(173))

##### 237 appointments were both:
##### booked 45–60 days in advance, and
##### located 15+ km from the clinic.
##### 173 of those 237 appointments were no-shows.

##### That means the no-show rate for this combined segment is:

##### 173 ÷ 237 = 73.00%

##### So this is a particularly high-risk segment.

##### Why this matters

##### Compare:

##### Overall clinic no-show rate: 48.46%
##### 45–60 day group: 67.23%
##### 45–60 days + 15+ km: 73.00%

##### This suggests that long lead time and greater distance may combine to identify an especially high-risk group.

##### Let's calculate how much this combined segment contributes to all 2,423 no-shows.

In [18]:
high_risk_no_show_share = (
    173 / (df_clean["appointment_outcome"] == "No-Show").sum()
) * 100

round(high_risk_no_show_share, 2)

np.float64(7.14)

##### The 45–60 day / 15+ km segment had a 73.00% no-show rate and accounted for 7.14% of all observed no-shows.

##### let's check whether previous appointment history adds even more risk to this high-risk segment.

In [19]:
high_risk_history = (
    high_risk_segment
    .groupby("previous_appointment_group")["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .round(2)
)

high_risk_history

previous_appointment_group
0 appointments      70.00
1-2 appointments    75.27
3-4 appointments    75.00
5+ appointments     64.29
Name: appointment_outcome, dtype: float64

In [20]:
high_risk_history_counts = (
    high_risk_segment
    .groupby("previous_appointment_group")
    .size()
)

high_risk_history_counts

previous_appointment_group
0 appointments      10
1-2 appointments    93
3-4 appointments    92
5+ appointments     42
dtype: int64

##### Within the highest-risk combination of 45–60 day booking lead time and 15+ km distance, no-show rates remained high across all previous-appointment groups. The 1–2 and 3–4 appointment groups recorded the highest observed rates at approximately 75%. However, because this is a relatively small combined segment (237 appointments), the interaction should be treated as an exploratory pattern rather than a definitive risk rule.

##### Let's now quantify the overall high-risk segment rate precisely so we can compare it with the overall 48.46% rate.

In [21]:
high_risk_rate = (
    (high_risk_segment["appointment_outcome"] == "No-Show").mean()
) * 100

round(high_risk_rate, 2)

np.float64(73.0)

In [22]:
high_risk_difference = high_risk_rate - 48.46

round(high_risk_difference, 2)

np.float64(24.54)

##### Long booking lead time combined with greater distance identifies a particularly high-risk appointment segment, with a 73.00% observed no-show rate—24.54 percentage points above the overall clinic rate.

## refining and validating the Week 5 KPIs

### verify the Total appointments

In [23]:
outcome_total = (
    (df_clean["appointment_outcome"] == "No-Show").sum()
    + (df_clean["appointment_outcome"] == "Attended").sum()
    + (df_clean["appointment_outcome"] == "Cancelled").sum()
)

outcome_total

np.int64(5000)

### verify the 48.46% overall no-show rate

In [24]:
validated_no_show_rate = (
    (df_clean["appointment_outcome"] == "No-Show").sum()
    / len(df_clean)
) * 100

round(validated_no_show_rate, 2)

np.float64(48.46)

### 45–60 Day No-Show Rate

In [25]:
long_lead_rate = (
    (
        df_clean[df_clean["lead_time_group"] == "45-60 days"]["appointment_outcome"]
        == "No-Show"
    ).mean()
) * 100

round(long_lead_rate, 2)

np.float64(67.23)

### what percentage of all clinic no-shows came from this 45–60 day group

In [26]:
long_lead_no_show_share = (
    (
        (
            df_clean[df_clean["lead_time_group"] == "45-60 days"]["appointment_outcome"]
            == "No-Show"
        ).sum()
        / (df_clean["appointment_outcome"] == "No-Show").sum()
    ) * 100
)

round(long_lead_no_show_share, 2)

np.float64(34.71)

In [27]:
high_risk_no_show_rate = (
    (
        high_risk_segment["appointment_outcome"] == "No-Show"
    ).mean()
) * 100

round(high_risk_no_show_rate, 2)

np.float64(73.0)

In [28]:
week6_evidence = pd.DataFrame({
    "Finding": [
        "Booking lead time",
        "Distance to clinic",
        "Previous appointment history",
        "Reminder status"
    ],
    "Evidence": [
        "45–60 day no-show rate = 67.23%; χ² = 359.38; p < 0.001",
        "15+ km no-show rate = 54.09%; χ² = 16.21; p = 0.0010",
        "5+ previous appointments = 51.66%; χ² = 11.84; p = 0.0079",
        "Reminder = 47.36% vs no reminder = 51.39%; χ² = 6.30; p = 0.0120"
    ],
    "Business_Impact": [
        "34.71% of all no-shows came from the 45–60 day group",
        "Higher risk becomes more pronounced with long lead time",
        "Secondary risk indicator; effect is modest",
        "Lower observed no-show rate, especially for long-lead appointments"
    ],
    "Recommended_Action": [
        "Prioritise enhanced confirmation for 45–60 day appointments",
        "Prioritise long-lead appointments 15+ km from the clinic",
        "Use appointment history as a secondary risk indicator",
        "Maintain and test reminder strategies"
    ]
})

week6_evidence

,Finding,Evidence,Business_Impact,Recommended_Action
0,Booking lead time,45–60 day no-show rate = 67.23%; χ² = 359.38; ...,34.71% of all no-shows came from the 45–60 day...,Prioritise enhanced confirmation for 45–60 day...
1,Distance to clinic,15+ km no-show rate = 54.09%; χ² = 16.21; p = ...,Higher risk becomes more pronounced with long ...,Prioritise long-lead appointments 15+ km from ...
2,Previous appointment history,5+ previous appointments = 51.66%; χ² = 11.84;...,Secondary risk indicator; effect is modest,Use appointment history as a secondary risk in...
3,Reminder status,Reminder = 47.36% vs no reminder = 51.39%; χ² ...,"Lower observed no-show rate, especially for lo...",Maintain and test reminder strategies


In [29]:
week6_evidence.dtypes

Finding               str
Evidence              str
Business_Impact       str
Recommended_Action    str
dtype: object

# Data Science cross-track contribution

##### Create the modelling target

In [30]:
df_clean["no_show_target"] = (
    df_clean["appointment_outcome"] == "No-Show"
).astype(int)

df_clean["no_show_target"].value_counts()

no_show_target
0    2577
1    2423
Name: count, dtype: int64

##### candidate predictors already supported by our Week 6 analysis

In [31]:
candidate_features = [
    "booking_lead_days",
    "previous_appointments",
    "previous_no_shows",
    "distance_to_clinic_km",
    "reminder_sent",
    "reminder_channel",
    "appointment_type",
    "age_group",
    "appointment_time",
    "appointment_day"
]

candidate_features

['booking_lead_days',
 'previous_appointments',
 'previous_no_shows',
 'distance_to_clinic_km',
 'reminder_sent',
 'reminder_channel',
 'appointment_type',
 'age_group',
 'appointment_time',
 'appointment_day']

##### check for missing values

In [32]:
df_clean[candidate_features].isnull().sum()

booking_lead_days           0
previous_appointments       0
previous_no_shows           0
distance_to_clinic_km      90
reminder_sent               0
reminder_channel         1366
appointment_type            0
age_group                   0
appointment_time            0
appointment_day             0
dtype: int64

##### reminder_channel having 1,366 missing values is not necessarily a data problem. We already established that these correspond to appointments where: reminder_sent = No

In [33]:
pd.crosstab(
    df_clean["reminder_sent"],
    df_clean["reminder_channel"].isnull()
)

reminder_channel,False,True
reminder_sent,,
No,0,1366
Yes,3634,0


##### When reminder_sent = No, reminder_channel is always missing. When reminder_sent = Yes, reminder_channel is always populated. Therefore, the 1,366 missing values in reminder_channel are structural missingness, not data-quality errors.

In [34]:
df_clean["reminder_channel"].value_counts(dropna=False)

reminder_channel
SMS         2000
NaN         1366
WhatsApp    1101
Email        533
Name: count, dtype: int64

##### So the missing values actually mean: No reminder was sent.

In [35]:
feature_summary = df_clean[candidate_features].dtypes

feature_summary

booking_lead_days          int64
previous_appointments      int64
previous_no_shows          int64
distance_to_clinic_km    float64
reminder_sent                str
reminder_channel             str
appointment_type             str
age_group                    str
appointment_time             str
appointment_day              str
dtype: object

In [36]:
pd.crosstab(
    df_clean["distance_to_clinic_km"].isnull(),
    df_clean["appointment_outcome"]
)

appointment_outcome,Attended,Cancelled,No-Show
distance_to_clinic_km,,,
False,2275,259,2376
True,39,4,47


In [37]:
distance_missing_pct = (
    df_clean["distance_to_clinic_km"].isnull().mean()
) * 100

round(distance_missing_pct, 2)

np.float64(1.8)

##### Distance-to-clinic data has 90 missing values (1.8% of appointments). The missing-distance group had a 52.22% no-show rate compared with 48.39% among records with recorded distance. Because the missing proportion is small, records will be retained and missing distance values handled during model preparation.

## preparing the modelling version of reminder_channel

In [38]:
df_clean["reminder_channel"].unique()

<ArrowStringArray>
['WhatsApp', 'SMS', 'Email', nan]
Length: 4, dtype: str

##### For the eventual modelling dataset: NaN will be converted to No Reminder, giving us four meaningful categories.

In [39]:
df_clean["reminder_sent"].unique()

<ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str

In [40]:
df_clean["reminder_channel_model"] = (
    df_clean["reminder_channel"].fillna("No Reminder")
)

In [41]:
df_clean["reminder_channel_model"].value_counts()

reminder_channel_model
SMS            2000
No Reminder    1366
WhatsApp       1101
Email           533
Name: count, dtype: int64

In [42]:
df_clean["no_show_target"].isnull().sum()

np.int64(0)

##### means there are no missing values in no_show_target

In [43]:
df_clean["reminder_channel_model"].isnull().sum()

np.int64(0)

##### confirms that reminder_channel_model has no missing values.

### checking the numerical predictors

In [44]:
df_clean[
    [
        "booking_lead_days",
        "previous_appointments",
        "previous_no_shows",
        "distance_to_clinic_km"
    ]
].describe()

,booking_lead_days,previous_appointments,previous_no_shows,distance_to_clinic_km
count,5000.00000,5000.000000,5000.000000,4910.000000
mean,29.63860,3.013800,0.544200,10.109572
std,17.39936,1.741211,0.746832,6.590030
min,0.00000,0.000000,0.000000,0.500000
25%,15.00000,2.000000,0.000000,5.300000
50%,30.00000,3.000000,0.000000,8.700000
75%,45.00000,4.000000,1.000000,13.500000
max,60.00000,11.000000,5.000000,45.000000


#### checking whether previous_no_shows can ever be greater than previous_appointments in the modelling data

In [45]:
(
    df_clean["previous_no_shows"]
    > df_clean["previous_appointments"]
).sum()

np.int64(0)

##### confirms that there are no records where previous no-shows exceed previous appointments.

##### Does previous_no_shows provide information beyond simply knowing the number of previous appointments?

In [46]:
pd.crosstab(
    df_clean["previous_no_shows"],
    df_clean["previous_appointments"]
)

previous_appointments,0,1,2,3,4,5,6,7,8,9,10,11
previous_no_shows,,,,,,,,,,,,
0,242,635,766,604,403,170,71,24,5,1,0,0
1,0,114,328,393,322,221,102,42,17,8,1,0
2,0,0,37,84,123,92,49,37,12,3,1,0
3,0,0,0,5,12,18,15,17,9,1,0,1
4,0,0,0,0,1,1,4,3,3,0,0,0
5,0,0,0,0,0,0,0,1,2,0,0,0


##### previous_appointments and previous_no_shows are related features because previous no-shows cannot exceed previous appointments. Both are retained as candidate predictors, with redundancy/multicollinearity to be assessed during model development.

In [47]:
previous_no_show_rate = (
    df_clean
    .groupby("previous_no_shows")["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .round(2)
)

previous_no_show_rate

previous_no_shows
0     43.51
1     53.49
2     59.36
3     67.95
4     66.67
5    100.00
Name: appointment_outcome, dtype: float64

##### Previous no-show behaviour appears to be a more directly relevant behavioural indicator than total previous appointments, although its predictive strength should be formally evaluated during modelling.

In [48]:
previous_no_show_table = pd.crosstab(
    df_clean["previous_no_shows"],
    df_clean["appointment_outcome"] == "No-Show"
)

previous_no_show_table

appointment_outcome,False,True
previous_no_shows,,
0,1650,1271
1,720,828
2,178,260
3,25,53
4,4,8
5,0,3


In [49]:
chi2, p_value, dof, expected = chi2_contingency(
    previous_no_show_table
)

chi2, p_value

(np.float64(81.78076021695519), np.float64(3.557630961982681e-16))

In [50]:
pd.crosstab(
    df_clean["previous_appointments"],
    df_clean["previous_no_shows"]
)

previous_no_shows,0,1,2,3,4,5
previous_appointments,,,,,,
0,242,0,0,0,0,0
1,635,114,0,0,0,0
2,766,328,37,0,0,0
3,604,393,84,5,0,0
4,403,322,123,12,1,0
5,170,221,92,18,1,0
6,71,102,49,15,4,0
7,24,42,37,17,3,1
8,5,17,12,9,3,2


##### Previous no-shows appear to be a more behaviourally direct indicator of future no-show risk than simply counting previous appointments.

##### quantify the relationship between the number of previous appointments and previous no-shows

In [51]:
df_clean[
    ["previous_appointments", "previous_no_shows"]
].corr()

,previous_appointments,previous_no_shows
previous_appointments,1.000000,0.458792
previous_no_shows,0.458792,1.000000


##### correlation is: r = 0.459. This indicates a moderate positive relationship between: previous_appointments and previous_no_shows
##### Keep both previous_appointments and previous_no_shows as candidate predictors, but evaluate their individual contribution during modelling.

##### checking whether previous no-shows add information within different lead-time groups

In [52]:
previous_no_show_lead_time = (
    df_clean
    .groupby(["previous_no_shows", "lead_time_group"])["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .unstack()
    .round(2)
)

previous_no_show_lead_time

lead_time_group,0-14 days,15-29 days,30-44 days,45-60 days
previous_no_shows,,,,
0,26.74,37.94,48.45,61.30
1,32.56,47.37,58.25,72.89
2,47.54,50.85,62.11,80.58
3,31.82,85.00,66.67,90.48
4,50.00,75.00,33.33,100.00
5,100.00,NaN,100.00,100.00


##### Booking lead time is not simply appearing important because certain types of patients have more previous no-shows. It remains strongly associated with no-shows across behavioural-risk groups.

##### checking how many appointments are actually in each previous-no-show × lead-time combination

In [53]:
previous_no_show_lead_time_counts = (
    df_clean
    .groupby(["previous_no_shows", "lead_time_group"])
    .size()
    .unstack()
)

previous_no_show_lead_time_counts

lead_time_group,0-14 days,15-29 days,30-44 days,45-60 days
previous_no_shows,,,,
0,748.0,709.0,743.0,721.0
1,347.0,399.0,400.0,402.0
2,122.0,118.0,95.0,103.0
3,22.0,20.0,15.0,21.0
4,2.0,4.0,3.0,3.0
5,1.0,NaN,1.0,1.0


#### Both booking lead time and previous no-show behaviour show elevated no-show rates when considered together, suggesting that combining these variables could improve identification of higher-risk appointments.

##### determining whether previous no-shows and lead time are themselves related

In [54]:
df_clean[
    ["booking_lead_days", "previous_no_shows"]
].corr()

,booking_lead_days,previous_no_shows
booking_lead_days,1.000000,0.000206
previous_no_shows,0.000206,1.000000


##### booking_lead_days and previous_no_shows should both be retained as candidate predictors because they show strong associations with the no-show target while exhibiting essentially no linear correlation with each other (r = 0.0002).

##### analyzing is Previous No-Shows × Distance to Clinic

In [55]:
previous_no_show_distance = (
    df_clean
    .groupby(["previous_no_shows", "distance_group"])["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .unstack()
    .round(2)
)

previous_no_show_distance

distance_group,0-5 km,10-15 km,15+ km,5-10 km
previous_no_shows,,,,
0,42.14,43.22,48.52,41.68
1,50.60,54.43,59.86,51.16
2,53.70,60.71,69.23,56.82
3,94.12,57.89,56.25,66.67
4,0.00,50.00,100.00,71.43
5,100.00,100.00,NaN,100.00


In [56]:
previous_no_show_distance_table = pd.crosstab(
    df_clean["previous_no_shows"],
    df_clean["appointment_outcome"] == "No-Show"
)

previous_no_show_distance_table

appointment_outcome,False,True
previous_no_shows,,
0,1650,1271
1,720,828
2,178,260
3,25,53
4,4,8
5,0,3


##### Previous no-show behaviour is a strong behavioural indicator, while distance provides additional segmentation information. Their combination can help identify higher-risk appointments, but the combined interaction should be treated as exploratory.

##### previous no-shows × reminder status.

In [57]:
previous_no_show_reminder = (
    df_clean
    .groupby(["previous_no_shows", "reminder_sent"])["appointment_outcome"]
    .apply(lambda x: (x == "No-Show").mean() * 100)
    .unstack()
    .round(2)
)

previous_no_show_reminder

reminder_sent,No,Yes
previous_no_shows,,
0,45.44,42.80
1,56.54,52.32
2,66.67,56.51
3,70.00,67.24
4,100.00,42.86
5,NaN,100.00


##### The lower observed no-show rate associated with reminders persists across levels of previous no-show behaviour, suggesting that reminder status may provide useful additional information for risk segmentation. However, causal effectiveness should be tested rather than assumed.

In [58]:
df_clean.columns.tolist()

['appointment_id',
 'patient_id',
 'gender',
 'age',
 'age_group',
 'appointment_type',
 'booking_date',
 'appointment_date',
 'appointment_day',
 'appointment_time',
 'booking_lead_days',
 'previous_appointments',
 'previous_no_shows',
 'reminder_sent',
 'reminder_channel',
 'distance_to_clinic_km',
 'waiting_time_minutes',
 'appointment_outcome',
 'lead_time_group',
 'previous_appointment_group',
 'distance_group',
 'waiting_time_group',
 'no_show_target',
 'reminder_channel_model']

In [59]:
df_clean.to_csv("HealthConnect_clean.csv", index=False)